# Byte Pair Encoding for LLMs

## Unicode code points

In [1]:
text = """Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."""
print(text)
print(f"Length: {len(text)}")

Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
Length: 533


In [2]:
chr_text = list(ord(c) for c in text)
print(chr_text)
print(f"Length: {len(chr_text)}")

[65333, 65358, 65353, 65347, 65359, 65348, 65349, 33, 32, 127332, 127325, 127320, 127314, 127326, 127315, 127316, 8253, 32, 127482, 8204, 127475, 8204, 127470, 8204, 127464, 8204, 127476, 8204, 127465, 8204, 127466, 33, 32, 128516, 32, 84, 104, 101, 32, 118, 101, 114, 121, 32, 110, 97, 109, 101, 32, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 101, 32, 105, 110, 116, 111, 32, 116, 104, 101, 32, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 101, 32, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 101, 32, 111, 117, 103, 104, 116, 32, 116, 111, 32, 8220, 115, 117, 112, 112, 111, 114, 116, 32, 85, 110, 105, 99, 111, 100, 101, 8221, 32, 105, 110, 32, 111, 117, 114, 32, 115, 111, 102, 116, 119, 97, 114, 101, 32, 40, 119, 104, 97, 116, 101, 118, 101, 114, 32, 116, 104, 97, 116, 32, 109, 101, 97, 110, 115, 8212, 108, 105, 107, 101, 32, 

In [3]:
bytes_text = list(text.encode("utf-8"))
print(bytes_text)
print(f"Length: {len(bytes_text)}")

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 101, 32, 118, 101, 114, 121, 32, 110, 97, 109, 101, 32, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 101, 32, 105, 110, 116, 111, 32, 116, 104, 101, 32, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 101, 32, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 101, 32, 111, 117, 103, 104, 116, 32, 116

In [4]:
s = list("😄".encode())
s

[240, 159, 152, 132]

In [5]:
byte_values = [240, 159, 152, 132]
bytes(byte_values)

b'\xf0\x9f\x98\x84'

In [6]:
bytes(byte_values).decode("utf-8")

'😄'

In [7]:
print(ord(" "))
print(ord("T"))
print(ord("😄"))   # maps character to unicode code point
print(chr(128516)) # maps from unicode code point to character

32
84
128516
😄


## Byte Pair Encoding

In [8]:
from collections import defaultdict

def get_stats(byte_seq: list[int]) -> dict:
    counts = defaultdict(int)
    for i in range(1, len(byte_seq)):
        pair = (byte_seq[i - 1], byte_seq[i])
        counts[pair] += 1
    return counts
counts = get_stats(bytes_text)
print(counts)

defaultdict(<class 'int'>, {(239, 188): 1, (188, 181): 1, (181, 239): 1, (239, 189): 6, (189, 142): 1, (142, 239): 1, (189, 137): 1, (137, 239): 1, (189, 131): 1, (131, 239): 1, (189, 143): 1, (143, 239): 1, (189, 132): 1, (132, 239): 1, (189, 133): 1, (133, 33): 1, (33, 32): 2, (32, 240): 3, (240, 159): 15, (159, 133): 7, (133, 164): 1, (164, 240): 1, (133, 157): 1, (157, 240): 1, (133, 152): 1, (152, 240): 1, (133, 146): 1, (146, 240): 1, (133, 158): 1, (158, 240): 1, (133, 147): 1, (147, 240): 1, (133, 148): 1, (148, 226): 1, (226, 128): 12, (128, 189): 1, (189, 32): 1, (159, 135): 7, (135, 186): 1, (186, 226): 1, (128, 140): 6, (140, 240): 6, (135, 179): 1, (179, 226): 1, (135, 174): 1, (174, 226): 1, (135, 168): 1, (168, 226): 1, (135, 180): 1, (180, 226): 1, (135, 169): 1, (169, 226): 1, (135, 170): 1, (170, 33): 1, (159, 152): 1, (152, 132): 1, (132, 32): 1, (32, 84): 1, (84, 104): 1, (104, 101): 6, (101, 32): 20, (32, 118): 1, (118, 101): 3, (101, 114): 6, (114, 121): 2, (121, 

In [9]:
def sort_stats(stats: dict) -> list[tuple[int, tuple[int, int]]]:
    return sorted(((v, k) for k, v in stats.items()), reverse=True)


In [10]:
sorted_stats = sorted(((v, k) for k, v in counts.items()), reverse=True)
print(sorted_stats)

[(20, (101, 32)), (15, (240, 159)), (12, (226, 128)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (159, 135)), (7, (159, 133)), (7, (97, 114)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (110, 32)), (5, (100, 101)), (5, (44, 32)), (5, (32, 115)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (111, 100)), (4, (110, 116)), (4, (110, 105)), (4, (105, 99)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (99, 111)), (4, (97, 109)), (4, (85, 110)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (4, (32, 85)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (

In [11]:
top1 = max(counts, key=counts.get)
print(f"Most frequent pair is {top1} with frequency of {counts[top1]}")

Most frequent pair is (101, 32) with frequency of 20


In [12]:
def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

In [13]:
merged = merge(bytes_text, top1, 256)
print(merged)
print(f"Length: {len(merged)}")

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 32, 226, 128, 156

In [14]:
bin(118)

'0b1110110'

In [15]:
def bpe(ids: list[int], n_steps: int) -> list[int]:
    idx = 256
    merges = {}
    for _ in range(n_steps):
        stats = get_stats(ids)
        top_pair = max(stats, key=stats.get)
        print(f"Merging {top_pair} into new token {idx}")
        ids = merge(ids, top_pair, idx)
        merges[top_pair] = idx
        idx += 1

    return ids, merges

merged, merges = bpe(bytes_text, 200)
print(merged)
print(f"Length: {len(merged)}")
print(f"Compression ratio: {len(bytes_text) / len(merged):.2f}")

Merging (101, 32) into new token 256
Merging (240, 159) into new token 257
Merging (226, 128) into new token 258
Merging (105, 110) into new token 259
Merging (115, 32) into new token 260
Merging (97, 110) into new token 261
Merging (116, 104) into new token 262
Merging (257, 133) into new token 263
Merging (257, 135) into new token 264
Merging (97, 114) into new token 265
Merging (239, 189) into new token 266
Merging (258, 140) into new token 267
Merging (267, 264) into new token 268
Merging (101, 114) into new token 269
Merging (111, 114) into new token 270
Merging (116, 32) into new token 271
Merging (259, 103) into new token 272
Merging (115, 116) into new token 273
Merging (261, 100) into new token 274
Merging (32, 262) into new token 275
Merging (44, 32) into new token 276
Merging (97, 109) into new token 277
Merging (275, 256) into new token 278
Merging (111, 117) into new token 279
Merging (85, 110) into new token 280
Merging (280, 105) into new token 281
Merging (281, 99) into

In [16]:
list(text.encode("utf-8")) == list(map(int, text.encode("utf-8")))

True

In [17]:
merges

{(101, 32): 256,
 (240, 159): 257,
 (226, 128): 258,
 (105, 110): 259,
 (115, 32): 260,
 (97, 110): 261,
 (116, 104): 262,
 (257, 133): 263,
 (257, 135): 264,
 (97, 114): 265,
 (239, 189): 266,
 (258, 140): 267,
 (267, 264): 268,
 (101, 114): 269,
 (111, 114): 270,
 (116, 32): 271,
 (259, 103): 272,
 (115, 116): 273,
 (261, 100): 274,
 (32, 262): 275,
 (44, 32): 276,
 (97, 109): 277,
 (275, 256): 278,
 (111, 117): 279,
 (85, 110): 280,
 (280, 105): 281,
 (281, 99): 282,
 (282, 111): 283,
 (283, 100): 284,
 (115, 276): 285,
 (273, 114): 286,
 (101, 265): 287,
 (274, 32): 288,
 (259, 116): 289,
 (111, 102): 290,
 (46, 32): 291,
 (108, 108): 292,
 (272, 32): 293,
 (261, 32): 294,
 (101, 110): 295,
 (33, 32): 296,
 (118, 269): 297,
 (121, 32): 298,
 (277, 256): 299,
 (105, 107): 300,
 (101, 260): 301,
 (119, 256): 302,
 (289, 111): 303,
 (303, 278): 304,
 (116, 260): 305,
 (290, 32): 306,
 (112, 114): 307,
 (307, 111): 308,
 (308, 103): 309,
 (309, 114): 310,
 (310, 277): 311,
 (311, 109):

In [18]:
bytes([65]).decode()

'A'

In [19]:
# Token id to bytes object for that token
vocab = {idx: bytes([idx]) for idx in range(256)}
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [20]:
bytes([65]) + bytes([66])

b'AB'

In [21]:
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [22]:
# tokens -> strings
def decode(tokens: list[int]) -> str:
    tokens = b"".join(vocab[t] for t in tokens)
    return tokens.decode("utf-8", errors="replace")
print(decode(merged))

Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.


In [23]:
print(decode([65, 66]))

AB


In [24]:
# strings -> tokens
def encode(text: str) -> list[int]:
    tokens = text.encode("utf-8")       # raw bytes
    tokens = list(map(int, tokens))     # list of integers (bytes)
    stats = get_stats(tokens)
    top1 = max(stats, key=stats.get)
    while top1 in merges:
        tokens = merge(tokens, top1, merges[top1])
        stats = get_stats(tokens)
        top1 = max(stats, key=stats.get)
    return tokens

In [25]:
decode(encode(text))

'Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺\u200c🇳\u200c🇮\u200c🇨\u200c🇴\u200c🇩\u200c🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.'